In [6]:
!pip install -q sentence-transformers faiss-cpu transformers torch

To avoid the warning about unauthenticated requests to Hugging Face Hub and potentially improve rate limits, you can set an `HF_TOKEN`.

1.  **Obtain an HF Token:** Go to [Hugging Face Settings](https://huggingface.co/settings/tokens) and create a new token.
2.  **Store in Colab Secrets:** In Colab, click on the "🔑" icon in the left sidebar, add a new secret, name it `HF_TOKEN`, and paste your token there. Make sure "Notebook access" is enabled for this notebook.
3.  **Use the token:** You can then load the token from Colab secrets in your code.

In [7]:
# Import `userdata` to securely access your HF_TOKEN from Colab secrets
from google.colab import userdata
import os

# Get the token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set the HF_TOKEN environment variable. This will be automatically picked up by Hugging Face libraries.
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN successfully loaded from Colab secrets and set.")
else:
    print("HF_TOKEN not found in Colab secrets. Please add it.")


HF_TOKEN successfully loaded from Colab secrets and set.


After running the above cell, rerun the RAG system cell (`4iEBFuvOlIBc`) to see if the warning is gone. The `transformers` library should automatically pick up the `HF_TOKEN` environment variable.

In [10]:
import faiss
import numpy as np
import os
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Install faiss-cpu if not already installed (to prevent ModuleNotFoundError)
!pip install -q faiss-cpu

# Get the token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set the HF_TOKEN environment variable. This will be automatically picked up by Hugging Face libraries.
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN successfully loaded from Colab secrets and set within this cell.")
else:
    print("HF_TOKEN not found in Colab secrets. Please add it to Colab secrets.")

# ---------- 1. Knowledge Base ----------
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# ---------- 2. Generate Embeddings ----------
# Pass the hf_token explicitly
embed_model = SentenceTransformer("all-MiniLM-L6-v2", use_auth_token=hf_token)
doc_embeddings = embed_model.encode(documents)

# ---------- 3. Build FAISS Vector Index ----------
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# ---------- 4. Query ----------
query = "What is RAG in AI?"

query_embedding = embed_model.encode([query])

# Retrieve top-2 relevant documents
D, I = index.search(
    np.array(query_embedding),
    k=2
)

retrieved_chunks = [documents[i] for i in I[0]]

# ---------- 5. Create Augmented Prompt ----------
context = " ".join(retrieved_chunks)

prompt = f"""Context: {context}
Question: {query}
Answer:"""

# ---------- 6. Generate Grounded Answer ----------
# Load tokenizer and model explicitly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base", token=hf_token)
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", token=hf_token)

# Directly use model.generate() for sequence-to-sequence models instead of `pipeline`
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
output_ids = model.generate(input_ids, max_length=60, num_beams=5, early_stopping=True) # Added num_beams and early_stopping for better generation
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Format the output to match the previous pipeline structure for consistency
answer = [{"generated_text": generated_text}]

# ---------- 7. Display Results ----------
print("========== RAG SYSTEM ==========")

print("\nQuery:")
print(query)

print("\nRetrieved Context:")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}")

print("\nGenerated Answer:")
print(answer[0]["generated_text"])

# ---------- Result ----------
print("\n========== RESULT ==========")
print("A Retrieval-Augmented Generation system was successfully")
print("implemented using FAISS as the vector database, producing")
print("answers grounded in retrieved document context.")


HF_TOKEN successfully loaded from Colab secrets and set within this cell.


/usr/local/lib/python3.12/dist-packages/sentence_transformers/sentence_transformer/model.py:178: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in a future release of SentenceTransformers.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


========== RAG SYSTEM ==========

Query:
What is RAG in AI?

Retrieved Context:
1. Python is a popular high-level programming language used in AI development.
2. Retrieval-Augmented Generation combines document retrieval with text generation.

Generated Answer:
document retrieval with text generation

========== RESULT ==========
A Retrieval-Augmented Generation system was successfully
implemented using FAISS as the vector database, producing
answers grounded in retrieved document context.
